In [ ]:


import pandas as pd


spec = pd.read_csv("atmospheric_spectroscopy.csv")

sys = pd.read_csv("planetary_systems.csv", comment="#", low_memory=False)


sys_default = sys[sys["default_flag"] == 1].copy()

sys_default = sys_default.drop_duplicates(subset="pl_name")

merged = spec.merge(
    sys_default,
    left_on="PL_NAME",
    right_on="pl_name",
    how="left",          
    suffixes=("_spec", "_sys"),
)

print(f"Spectroscopy rows:      {len(spec)}")
print(f"Unique planets (spec):  {spec['PL_NAME'].nunique()}")
print(f"Planetary systems rows: {len(sys_default)}")
print(f"Merged rows:            {len(merged)}")
unmatched = merged[merged["pl_name"].isna()]["PL_NAME"].unique()
print(f"Unmatched planet names: {list(unmatched)}")


merged.to_csv("merged_dataset.csv", index=False)
print("Saved merged_dataset.csv")

In [ ]:


import math
import sys
import pandas as pd



EARTH_RADIUS = 1.0          
EARTH_DENSITY = 5.51       
EARTH_ESC_VEL = 11.19       
EARTH_TEMP = 288.0         

WEIGHTS = {
    "radius": 0.57,
    "density": 1.07,
    "esc_vel": 0.70,
    "temp": 5.58,
}
N_PROPERTIES = 4


def esi_component(value, reference, weight):
    if pd.isna(value) or value <= 0:
        return None
    ratio_term = abs(value - reference) / (value + reference)
    return (1 - ratio_term) ** (weight / N_PROPERTIES)


def compute_escape_velocity_ratio(mass_earth, radius_earth):
    """Escape velocity relative to Earth: v_esc / v_esc_earth = sqrt(M/R) in Earth units."""
    if pd.isna(mass_earth) or pd.isna(radius_earth) or radius_earth <= 0:
        return None
    return math.sqrt(mass_earth / radius_earth) * EARTH_ESC_VEL


def compute_esi(row):
    radius = row.get("pl_rade")
    density = row.get("pl_dens")
    mass = row.get("pl_masse")
    temp = row.get("pl_eqt")

    esc_vel = compute_escape_velocity_ratio(mass, radius)

    esi_radius = esi_component(radius, EARTH_RADIUS, WEIGHTS["radius"])
    esi_density = esi_component(density, EARTH_DENSITY, WEIGHTS["density"])
    esi_escvel = esi_component(esc_vel, EARTH_ESC_VEL, WEIGHTS["esc_vel"])
    esi_temp = esi_component(temp, EARTH_TEMP, WEIGHTS["temp"])

    interior_parts = [x for x in (esi_radius, esi_density) if x is not None]
    surface_parts = [x for x in (esi_escvel, esi_temp) if x is not None]

    if not interior_parts or not surface_parts:
        return None

    esi_interior = math.prod(interior_parts) ** (1 / len(interior_parts))
    esi_surface = math.prod(surface_parts) ** (1 / len(surface_parts))

    return math.sqrt(esi_interior * esi_surface)


def esi_to_category(esi):
    if esi is None:
        return "Insufficient Data"
    if esi >= 0.8:
        return "Earth-like (High Habitability Potential)"
    elif esi >= 0.6:
        return "Moderately Habitable"
    elif esi >= 0.4:
        return "Marginally Habitable"
    else:
        return "Non-Habitable"


def classify_planet_type(radius, mass):
    """Standard radius/mass-based exoplanet classification bins used across exoplanet science."""
    if pd.isna(radius):
        return "Unknown"
    if radius < 1.5:
        return "Rocky"
    elif radius < 2.0:
        return "Super-Earth"
    elif radius < 4.0:
        return "Sub-Neptune"
    elif radius < 10.0:
        return "Neptune-like"
    else:
        return "Gas Giant"


def infer_atmosphere(planet_type, temp):
    
    if planet_type == "Unknown":
        return "Unknown (insufficient data)"

    if planet_type == "Gas Giant":
        return "Inferred: H/He-dominated primordial envelope"

    if planet_type == "Neptune-like":
        return "Inferred: H/He + volatile ices (Neptune-like envelope)"

    if planet_type == "Sub-Neptune":
        if pd.notna(temp) and temp > 800:
            return "Inferred: H/He envelope likely eroding (highly irradiated)"
        return "Inferred: possible H/He or steam/volatile-rich envelope"

    # Rocky or Super-Earth
    if pd.isna(temp):
        return "Inferred: thin secondary atmosphere (temperature unknown)"
    if temp > 1000:
        return "Inferred: atmosphere likely stripped / exotic vapor (extreme heat)"
    elif temp > 500:
        return "Inferred: thin or no atmosphere (too hot for stable volatiles)"
    elif 180 <= temp <= 320:
        return "Inferred: possible CO2/N2/H2O secondary atmosphere (temperate)"
    else:
        return "Inferred: thin atmosphere or frozen volatiles (cold)"

import sys
import pandas as pd
import numpy as np

def main():
    if len(sys.argv) != 3:
        print("Usage: python habitability_v2.py <input_csv> <output_csv>")
        sys.exit(1)

    input_path = sys.argv[1]
    output_path = sys.argv[2]

    df = pd.read_csv(input_path)

    # Compute Earth Similarity Index (ESI)
    
    esi_values: pd.Series = df.apply(compute_esi, axis=1)

    df["esi"] = esi_values
    df["habitability_category"] = esi_values.apply(esi_to_category)

    df["planet_type"] = df.apply(
        lambda row: classify_planet_type(
            row.get("pl_rade"),
            row.get("pl_masse"),
        ),
        axis=1,
    )

    df["inferred_atmosphere"] = df.apply(
        lambda row: infer_atmosphere(
            row["planet_type"],
            row.get("pl_eqt"),
        ),
        axis=1,
    )

    df.to_csv(output_path, index=False)

    print("\nHabitability category counts:")
    print(df["habitability_category"].value_counts(dropna=False).to_string())

    print("\nPlanet type counts:")
    print(df["planet_type"].value_counts(dropna=False).to_string())

    print("\nTop 10 by ESI:")

    top = (
        df.dropna(subset=["esi"])
          .sort_values("esi", ascending=False)
          .drop_duplicates(subset="pl_name")
          .head(10)
    )

    print(
        top[
            [
                "pl_name",
                "esi",
                "habitability_category",
                "planet_type",
                "inferred_atmosphere",
            ]
        ].to_string(index=False)
    )

    print(f"\nSaved to: {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb


def load_data(csv_path: str) -> pd.DataFrame:
    """
    Load exoplanet dataset.
    """
    csv_path = r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\Exoplanet_Habitable\exoplanet_details.csv"

    df = pd.read_csv(csv_path)

    print("=" * 60)
    print("Dataset Loaded Successfully")
    print("=" * 60)

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    print("\nColumn Names:\n")
    for col in df.columns:
        print(col)

    print("\nFirst 5 Rows:\n")
    print(df.head())

    return df

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd


def clean_data(df: pd.DataFrame):

    df = df.copy()

 

    identifier_columns = [
        "pl_name",
        "hostname",
        "hd_name",
        "hip_name",
        "tic_id",
        "gaia_id",
        "disc_facility",
        "disc_telescope",
        "disc_instrument",
        "disc_refname",
        "disc_pubdate",
        "disc_year",
        "pl_refname",
        "releasedate",
        "rowupdate",
        "rastr",
        "decstr",
        "pl_letter",
        "solution_id"
    ]

    existing = [c for c in identifier_columns if c in df.columns]

    df.drop(columns=existing, inplace=True)

    print(f"Dropped {len(existing)} identifier columns.")



    missing_ratio = df.isna().mean()

    high_missing = missing_ratio[missing_ratio > 0.60].index

    df.drop(columns=high_missing, inplace=True)

    print(f"Dropped {len(high_missing)} columns with >60% missing values.")



    numeric_columns = df.select_dtypes(
        include=["number"]
    ).columns

    numeric_imputer = SimpleImputer(strategy="median")

    df[numeric_columns] = numeric_imputer.fit_transform(
        df[numeric_columns]
    )

   

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in categorical_columns:
        df[col] = df[col].fillna("Unknown").astype(str)

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    if len(categorical_columns) > 0:
        df[categorical_columns] = encoder.fit_transform(
            df[categorical_columns]
        )

    print("\nCleaning Complete")
    print("Final Shape:", df.shape)

    return df, encoder, numeric_imputer

df = load_data("exoplanet_details.csv")

df, encoder, numeric_imputer = clean_data(df)

X = df.drop(
    columns=[
        "habitability_category",
        "planet_type"
    ]
)

y_habitability = df["habitability_category"]

y_planet_type = df["planet_type"]


model_habitability = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_habitability.fit(
    X,
    y_habitability
)


model_planet_type = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="mlogloss"
)

model_planet_type.fit(
    X,
    y_planet_type
)



from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


habitability_predictions = model_habitability.predict(X)

habitability_accuracy = accuracy_score(
    y_habitability,
    habitability_predictions
)

print("\n" + "=" * 60)
print("Habitability Model")
print("=" * 60)

print(f"Accuracy : {habitability_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(
    y_habitability,
    habitability_predictions
))

print("\nConfusion Matrix")
print(confusion_matrix(
    y_habitability,
    habitability_predictions
))



planet_predictions = model_planet_type.predict(X)

planet_accuracy = accuracy_score(
    y_planet_type,
    planet_predictions
)

print("\n" + "=" * 60)
print("Planet Type Model")
print("=" * 60)

print(f"Accuracy : {planet_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(
    y_planet_type,
    planet_predictions
))

print("\nConfusion Matrix")
print(confusion_matrix(
    y_planet_type,
    planet_predictions
))


importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model_habitability.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 20 Important Features")
print(importance.head(20).to_string(index=False))



def predict_new_planet(new_planet_df):

    new_planet_df = new_planet_df.copy()

    numeric_cols = new_planet_df.select_dtypes(include=["number"]).columns
    categorical_cols = new_planet_df.select_dtypes(include=["object", "category"]).columns

    if len(numeric_cols) > 0:
        new_planet_df[numeric_cols] = numeric_imputer.transform(
            new_planet_df[numeric_cols]
        )

    if len(categorical_cols) > 0:
        for col in categorical_cols:
            new_planet_df[col] = new_planet_df[col].fillna("Unknown").astype(str)

        new_planet_df[categorical_cols] = encoder.transform(
            new_planet_df[categorical_cols]
        )

    habitability = model_habitability.predict(new_planet_df)

    planet_type = model_planet_type.predict(new_planet_df)

    return habitability, planet_type


import joblib

joblib.dump(
    model_habitability,
    "habitability_model.pkl"
)

joblib.dump(
    model_planet_type,
    "planet_type_model.pkl"
)

joblib.dump(
    encoder,
    "ordinal_encoder.pkl"
)

joblib.dump(
    numeric_imputer,
    "numeric_imputer.pkl"
)

print("\nModels saved successfully.")


import joblib
import pandas as pd

habitability_model = joblib.load("habitability_model.pkl")
planet_type_model = joblib.load("planet_type_model.pkl")

encoder = joblib.load("ordinal_encoder.pkl")
numeric_imputer = joblib.load("numeric_imputer.pkl")

# ==========================================================
# PREDICT A NEW PLANET
# ==========================================================

print("\n" + "=" * 60)
print("NEW PLANET PREDICTION")
print("=" * 60)

choice = input("Predict a new planet? (y/n): ").strip().lower()

if choice == "y":

    print("\nEnter values for the following features.\n")

    new_data = {}

    for column in X.columns:

        if column in numeric_columns:

            while True:
                try:
                    value = input(f"{column}: ")

                    if value == "":
                        new_data[column] = np.nan
                    else:
                        new_data[column] = float(value)

                    break

                except ValueError:
                    print("Please enter a numeric value.")

        else:

            value = input(f"{column}: ")

            if value == "":
                value = "Unknown"

            new_data[column] = value

    new_df = pd.DataFrame([new_data])

   

    numeric_cols = new_df.select_dtypes(include=["number"]).columns

    if len(numeric_cols) > 0:

        new_df[numeric_cols] = numeric_imputer.transform(
            new_df[numeric_cols]
        )


    categorical_cols = new_df.select_dtypes(
        include=["object", "category"]
    ).columns

    if len(categorical_cols) > 0:

        for col in categorical_cols:
            new_df[col] = new_df[col].fillna("Unknown").astype(str)

        new_df[categorical_cols] = encoder.transform(
            new_df[categorical_cols]
        )


    new_df = new_df[X.columns]



    habitability_prediction = model_habitability.predict(new_df)[0]

    planet_prediction = model_planet_type.predict(new_df)[0]

    habitability_labels = {
        0: "Non-Habitable",
        1: "Marginally Habitable",
        2: "Potentially Habitable",
        3: "Highly Habitable"
    }

    planet_labels = {
        0: "Unknown",
        1: "Rocky Planet",
        2: "Super Earth",
        3: "Mini Neptune",
        4: "Neptune-like",
        5: "Gas Giant"
    }

    print("\n" + "=" * 60)
    print("PREDICTION RESULT")
    print("=" * 60)

    print(
        "Habitability :",
        habitability_labels.get(
            int(habitability_prediction),
            habitability_prediction
        )
    )

    print(
        "Planet Type  :",
        planet_labels.get(
            int(planet_prediction),
            planet_prediction
        )
    )

else:

    print("\nPrediction skipped.")

In [ ]:
import base64

encoded_string = "bWFkZSBieSB2cnVzaGFiaC5zIHN3cw=="
print(base64.b64decode(encoded_string).decode('utf-8'))
